# 01 - Crawling, Pembersihan, dan Persiapan Label

Notebook ini menghasilkan data komentar yang siap diberi label. Label otomatis IndoBERT hanya dipakai sebagai **pre-label**, bukan ground truth. Label final tetap harus diverifikasi manusia.

## Cara menjalankan di VS Code

1. Buat environment: `python -m venv .venv`.
2. Aktifkan environment dan jalankan `pip install -r requirements-notebook.txt`.
3. Pilih kernel `.venv` melalui menu **Select Kernel** di kanan atas notebook.
4. Isi `YOUTUBE_API_KEY` melalui file `.env` atau environment variable.
5. Jalankan sel dari atas ke bawah.
6. Untuk IndoBERT, pastikan PyTorch yang terpasang mendukung CUDA dan sel pemeriksaan GPU menampilkan `cuda:0`.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Project root:', PROJECT_ROOT)

Project root: C:\Users\lenovo\Documents\Github\sentube_thesis


## Pemeriksaan GPU untuk IndoBERT

RTX 3050 digunakan otomatis sebagai `cuda:0` ketika PyTorch CUDA tersedia. TF-IDF, mRMR, dan SVM scikit-learn tetap menggunakan CPU.

In [ ]:
try:
    import torch
except ImportError:
    torch = None

if torch is None:
    TORCH_DEVICE = None
    print('PyTorch belum terpasang. Pre-label IndoBERT tidak dapat dijalankan.')
elif torch.cuda.is_available():
    TORCH_DEVICE = 0
    torch.backends.cudnn.benchmark = True
    print('Perangkat IndoBERT: cuda:0')
    print('GPU:', torch.cuda.get_device_name(TORCH_DEVICE))
    print('VRAM:', round(torch.cuda.get_device_properties(TORCH_DEVICE).total_memory / 1024**3, 2), 'GB')
    print('PyTorch CUDA:', torch.version.cuda)
else:
    TORCH_DEVICE = -1
    print('CUDA tidak tersedia. Periksa driver dan instalasi PyTorch CUDA.')

## Konfigurasi pengumpulan data

In [ ]:
KEYWORDS = [
    'coretax',
    'coretax DJP',
    'coretax pajak',
    'coretax error',
    'aplikasi coretax',
]
PUBLISHED_AFTER = '2024-12-01T00:00:00Z'
PUBLISHED_BEFORE = '2026-05-17T00:00:00Z'  # Eksklusif: mencakup hingga 16 Mei 2026
MAX_VIDEOS_PER_KEYWORD = 10
MAX_COMMENTS_PER_VIDEO = 200
MIN_WORDS = 5

YOUTUBE_API_KEY = os.getenv('YOUTUBE_API_KEY', '')
print('API key tersedia:', bool(YOUTUBE_API_KEY))

## Crawling YouTube

Setiap baris menyimpan `video_id` agar pemisahan data berbasis grup dapat dilakukan pada notebook training.

In [ ]:
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError


def search_videos(youtube, keyword: str) -> list[dict]:
    response = youtube.search().list(
        part='snippet',
        q=keyword,
        type='video',
        relevanceLanguage='id',
        maxResults=MAX_VIDEOS_PER_KEYWORD,
        publishedAfter=PUBLISHED_AFTER,
        publishedBefore=PUBLISHED_BEFORE,
    ).execute()
    return [
        {
            'keyword': keyword,
            'video_id': item['id']['videoId'],
            'video_title': item['snippet']['title'],
            'channel_title': item['snippet']['channelTitle'],
            'video_published_at': item['snippet']['publishedAt'],
        }
        for item in response.get('items', [])
    ]


def fetch_comments(youtube, video: dict) -> list[dict]:
    rows = []
    page_token = None
    while len(rows) < MAX_COMMENTS_PER_VIDEO:
        response = youtube.commentThreads().list(
            part='snippet',
            videoId=video['video_id'],
            maxResults=min(100, MAX_COMMENTS_PER_VIDEO - len(rows)),
            order='time',
            textFormat='plainText',
            pageToken=page_token,
        ).execute()
        for item in response.get('items', []):
            snippet = item['snippet']['topLevelComment']['snippet']
            rows.append({
                **video,
                'comment_id': item['snippet']['topLevelComment']['id'],
                'text': snippet['textDisplay'],
                'comment_published_at': snippet['publishedAt'],
                'like_count': snippet.get('likeCount', 0),
            })
        page_token = response.get('nextPageToken')
        if not page_token:
            break
    return rows


def crawl_youtube() -> pd.DataFrame:
    if not YOUTUBE_API_KEY:
        raise RuntimeError('Set environment variable YOUTUBE_API_KEY terlebih dahulu.')
    youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)
    videos = []
    for keyword in KEYWORDS:
        videos.extend(search_videos(youtube, keyword))
    unique_videos = {video['video_id']: video for video in videos}.values()

    comments = []
    for index, video in enumerate(unique_videos, start=1):
        try:
            comments.extend(fetch_comments(youtube, video))
        except HttpError as error:
            print(f"Lewati {video['video_id']}: {error.resp.status}")
        print(f'{index}: {video["video_id"]} - total komentar {len(comments)}')
    return pd.DataFrame(comments)

In [ ]:
RAW_PATH = RAW_DIR / 'youtube_comments_raw.csv'

if RAW_PATH.exists():
    raw_df = pd.read_csv(RAW_PATH)
    print('Memakai hasil crawling yang sudah ada:', RAW_PATH)
else:
    raw_df = crawl_youtube()
    raw_df.to_csv(RAW_PATH, index=False)

print('Jumlah data mentah:', len(raw_df))
display(raw_df.head())

## Audit dan pembersihan data

In [ ]:
audit = pd.Series({
    'jumlah_baris': len(raw_df),
    'teks_kosong': raw_df['text'].isna().sum(),
    'comment_id_duplikat': raw_df['comment_id'].duplicated().sum(),
    'teks_duplikat': raw_df['text'].fillna('').str.casefold().duplicated().sum(),
    'jumlah_video': raw_df['video_id'].nunique(),
})
display(audit.to_frame('nilai'))

In [ ]:
clean_df = raw_df.copy()
clean_df = clean_df.dropna(subset=['comment_id', 'video_id', 'text'])
clean_df['text'] = clean_df['text'].astype(str).str.strip()
clean_df = clean_df[clean_df['text'].str.split().str.len() >= MIN_WORDS]
clean_df['_dedup_text'] = (
    clean_df['text'].str.casefold().str.replace(r'\s+', ' ', regex=True).str.strip()
)
clean_df = clean_df.drop_duplicates(subset='comment_id')
clean_df = clean_df.drop_duplicates(subset='_dedup_text').drop(columns='_dedup_text')
clean_df['comment_published_at'] = pd.to_datetime(
    clean_df['comment_published_at'], utc=True, errors='coerce'
)
clean_df = clean_df.sort_values('comment_published_at').reset_index(drop=True)
print('Data setelah pembersihan:', len(clean_df))
display(clean_df.head())

## Pre-label opsional menggunakan IndoBERT

Isi `INDOBERT_SENTIMENT_MODEL` dengan checkpoint yang sudah di-fine-tune untuk tiga kelas. Periksa model card, urutan label, lisensi, dan domain data sebelum digunakan. Biarkan kosong jika pelabelan awal dilakukan manual.

In [ ]:
INDOBERT_SENTIMENT_MODEL = ''  # Contoh: organisasi/nama-checkpoint-yang-sudah-divalidasi
RUN_PRELABEL = bool(INDOBERT_SENTIMENT_MODEL)

if RUN_PRELABEL:
    if torch is None or TORCH_DEVICE != 0:
        raise RuntimeError(
            'Pre-label IndoBERT dikonfigurasi menggunakan RTX 3050, tetapi CUDA tidak tersedia.'
        )
    from transformers import pipeline

    classifier = pipeline(
        'text-classification',
        model=INDOBERT_SENTIMENT_MODEL,
        tokenizer=INDOBERT_SENTIMENT_MODEL,
        device=TORCH_DEVICE,
        dtype=torch.float16,
        truncation=True,
        top_k=None,
    )
    predictions = classifier(clean_df['text'].tolist(), batch_size=8)
    best = [max(items, key=lambda item: item['score']) for items in predictions]
    clean_df['label_model'] = [item['label'] for item in best]
    clean_df['model_confidence'] = [item['score'] for item in best]
else:
    clean_df['label_model'] = pd.NA
    clean_df['model_confidence'] = pd.NA
    print('Pre-label dilewati. Isi checkpoint jika diperlukan.')

## Ekspor lembar anotasi

Isi `label_annotator_1` dan `label_annotator_2` dengan `positif`, `netral`, atau `negatif`. Gunakan `needs_adjudication` untuk menandai perbedaan dan isi `label_final` setelah keputusan akhir.

In [ ]:
for column in ['label_annotator_1', 'label_annotator_2', 'needs_adjudication', 'label_final']:
    clean_df[column] = pd.NA

LABELING_PATH = PROCESSED_DIR / 'comments_for_labeling.csv'
clean_df.to_csv(LABELING_PATH, index=False)
print('Template anotasi:', LABELING_PATH)
display(clean_df[['comment_id', 'text', 'label_model', 'model_confidence', 'label_final']].head())

## Setelah anotasi selesai

1. Simpan hasil sebagai `data/processed/comments_labeled.csv`.
2. Pastikan kolom `label_final` tidak kosong.
3. Jangan mengubah teks mentah.
4. Lanjutkan ke `02_training_svm_mrmr.ipynb`.